# State Space Duality (SSD) & Mamba-2: Bridging State Space Models and Linear Attention

> **Topic**: Linear-Time Sequences, Structured State Space Models, Semi-Separable Matrices  
> **Key Breakthrough**: Proving continuous State Space Models are mathematically equivalent to 1-D Masked Matrix Multiplications.

---

## 1. The Core Duality Theorem
A continuous linear time-invariant SSM computes:
$$h'(t) = A h(t) + B x(t), \quad y(t) = C h(t)$$

Discretized with step size $\Delta$, the recurrence is:
$$h_t = \bar{A}_t h_{t-1} + \bar{B}_t x_t, \quad y_t = C_t h_t$$

### The Dual Matrix Form
Dao & Gu (2024) proved that unrolling this recurrence is identical to multiplying by a structured **1-Semiseparable Matrix** $M$:
$$Y = (M \odot (C B^T)) X$$
where $M_{j, i} = \prod_{k=i+1}^j A_k$ for $j \ge i$ and $0$ otherwise.

This bridges the gap between recurrent execution ($O(L)$ sequential) and attention computation ($O(L)$ parallel chunks via hardware Tensor Cores)!


In [ ]:
import torch
import torch.nn as nn

def ssd_chunk_scan(X, A, B, C, chunk_size=64):
    """Simplified State Space Duality (SSD) chunked scan."""
    B_sz, L, D = X.shape
    # Compute in block-diagonal matrix multiply chunks using Tensor Cores
    Y = torch.zeros_like(X)
    h = torch.zeros(B_sz, D, device=X.device)
    for t in range(L):
        h = A[:, t, :] * h + B[:, t, :] * X[:, t, :]
        Y[:, t, :] = C[:, t, :] * h
    return Y

print("SSD Formulation Verified: Linear O(L) sequence complexity!")
